# Utilizing Rule-Based Semantic Extraction and Text Parsing to Automate Scoring Cards in *Magic: The Gathering*
### IMT 575 - Spring 2026
Samuel Lee, Sitong Liu, Zach Greenman

In [1]:
import pandas as pd

creatures_full = pd.read_json('./data/creatures_full.json')

combined_points = pd.read_csv('./data/combined_csv_pts.csv')

scryfall_cols = [
    'oracle_id',
    'name',
    'set',
    'released_at',
    'cmc',
    'power',
    'toughness',
    'oracle_text',
    'keywords',
    'color_identity',
    'rarity'
]

creatures_full = creatures_full.drop_duplicates(
    subset=['oracle_id', 'set']
)

creatures_subset = creatures_full[scryfall_cols]

merged = combined_points.merge(
    creatures_subset,
    on=['name', 'set'],
    how='left',
    suffixes=('_old', '')
)

merged['released_at'] = pd.to_datetime(
    merged['released_at'],
    errors='coerce'
)

merged['year'] = merged['released_at'].dt.year

missing = merged['oracle_id'].isna().sum()

print(f'Missing oracle_id rows: {missing}')

Missing oracle_id rows: 0


In [2]:
merged = merged.rename(columns={
    'power_old': 'power_numeric',
    'toughness_old': 'toughness_numeric'
})

In [3]:
merged['power_numeric'] = pd.to_numeric(
    merged['power'],
    errors='coerce'
)

merged['toughness_numeric'] = pd.to_numeric(
    merged['toughness'],
    errors='coerce'
)

column_order = [
    'oracle_id',
    'name',
    'set',
    'set_name',
    'released_at',
    'year',

    'cmc',

    'power',
    'toughness',

    'power_numeric',
    'toughness_numeric',

    'keywords',
    'oracle_text',

    'color_identity',
    'rarity',

    'points'
]

merged = merged[column_order]

merged.to_csv(
    './data/combined_points_cleaned.csv',
    index=False
)

print('Saved combined_points_cleaned.csv tp /data')

Saved combined_points_cleaned.csv tp /data
